In [1]:
import pandas as pd
from collections import Counter
import os
import numpy as np
from tqdm import tqdm
import pickle
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import gcsfs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu

bucket = os.getenv("WORKSPACE_BUCKET")

def write_df_to_bucket(df, filename: str):
    # save mt directly to bucket
    df.to_csv(f"{bucket}/data/{filename}.csv", index = None)
def read_df_from_bucket(filename: str):
    # save mt directly to bucket
    return pd.read_csv(f"{bucket}/data/{filename}.csv")

fs = gcsfs.GCSFileSystem()
def save_dict(data_dict, filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "wb") as h:
        pickle.dump(data_dict, h)
def load_dict(filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "rb") as h:
        data_dict = pickle.load(h)
    return data_dict
def chunk_list(lst, chunk_size):
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

def count_label(data_dict):
    la = []
    for k, v in data_dict.items():
        la.append(v['label'])
    print(Counter(la))

In [6]:
# cancer cohort 
meds_data = read_df_from_bucket('meds_all_data_rev') # dataframe with columns of subject_id, prediction_time, label
person_ids = meds_data['subject_id'].tolist()

# clean data

In [17]:
def chunk_list(lst, chunk_size):
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

In [16]:
import pandas as pd
import pandas as pd
# from fun import *
from tqdm import tqdm
import numpy as np

class DataCleaner:
    def __init__(self, case, ctrl, col_name):
        self.case = case
        self.ctrl = ctrl
        self.col_name = col_name

    def cancer_cohort(self, cancer_ids):
        case = self.case[self.case[self.col_name]!=0] # remove unknown concept ids
        case = case[~case.duplicated()] # remove duplicates
        case = case[case['person_id'].isin(cancer_ids)] 
        # identify real cancer cases
        return case

    def ctrl_cohort(self):
        print('clean data')
        ctrl = self.ctrl[self.ctrl[self.col_name]!=0] # remove unknown concept ids
        ctrl = ctrl[~ctrl.duplicated()] # remove duplicates

        ctrl['condition_start_date'] = pd.to_datetime(ctrl['condition_start_date'], errors = 'coerce')

        # Step 1: Sort the DataFrame by 'person_id' and 'condition_start_datetime'
        print('sort data')
        ctrl = ctrl.sort_values(by=['person_id', 'condition_start_date'])

        # Calculate the last diagnosis date for each person_id minus two years
        print('cut off data - 2 years')
        last_diag_minus_two_years = ctrl.groupby('person_id')['condition_start_date'].transform('max') - pd.Timedelta(days=730)

        # Filter the DataFrame to only include dates before two years from the last diagnosis
        ctrl['within_two_years'] = ctrl['condition_start_date'] < last_diag_minus_two_years

        # Now, filter to keep only those within the two-year period
        ctrl = ctrl[ctrl['within_two_years']]

        # Step 2: Ensure at least 5 records exist for the condition within the timeframe for each person_id
        # This involves a groupby and filter operation
        print('filter data - 5 records')
        ctrl = ctrl.groupby('person_id').filter(lambda x: len(x) >= 5)

        return ctrl

    def cancer_categorization(self, cancer_diag, cancer_cat):
        cancer_diag['condition_start_date'] = pd.to_datetime(cancer_diag['condition_start_date'])
        ids = list(cancer_diag['person_id'].unique())
        grk = cancer_diag.groupby('person_id')
        cancer_cat_dict = dict(zip(cancer_cat['concept_name'], cancer_cat['cancer_type']))
        cancer_name_lst = list(cancer_cat['concept_name'].unique())
        rows = []
        for i in tqdm(ids):
            dat = grk.get_group(i)
            dat = dat.sort_values(by = 'condition_start_date')
            first_row = dat.iloc[0]
            if first_row['concept_name'] in cancer_name_lst:
                rows.append([i, first_row['condition_start_date'], first_row['concept_name'], cancer_cat_dict[first_row['concept_name']]])
            else: 
                rows.append([i, first_row['condition_start_date'], first_row['concept_name'], np.nan])
        df = pd.DataFrame(rows, columns=['person_id', 'condition_start_date', 'concept_name','cancer_type'])
        df_filtered = df[~df['concept_name'].str.contains('Secondary', na = False)]
        print('# of patients with first diagnosis secondary cancer: ', len(df)-len(df_filtered))
        return df_filtered
    
    def first_cancer(self, cancer_diag, concept):
        patids = list(cancer_diag['person_id'].unique())
        grk = cancer_diag.groupby('person_id')
        cancer_diag['condition_start_date'] = pd.to_datetime(cancer_diag['condition_start_date'], errors = 'coerce')

        rows = []
        for i in tqdm(patids):
            dat = grk.get_group(i)
            dat = dat.sort_values(by = 'condition_start_date')
            dat = dat.reset_index(drop = True)
            rows.append(dat.loc[0])
        first_diag = pd.DataFrame(rows)

        concept_dict = dict(zip(concept['concept_id'], concept['concept_name']))

        first_diag['concept_name'] = first_diag['condition_concept_id'].map(concept_dict)
        if first_diag['concept_name'].isna().any():
            print('check missing concept name')
        return first_diag
    
    def cancer_cohort_by_prediction_time(self, cancer_cond, first_diag, months_before=12):
        data = []
        cancer_cond['condition_start_date'] = pd.to_datetime(cancer_cond['condition_start_date'], errors = 'coerce')
        first_diag_dict = dict(zip(first_diag['person_id'], first_diag['condition_start_date']))
        patids = list(cancer_cond['person_id'].unique())
        grk = cancer_cond.groupby('person_id')
        for i in patids:
            dat = grk.get_group(i)
            dat = dat[dat['condition_start_date']<(first_diag_dict[i] - pd.DateOffset(months=months_before))]
            if len(dat)>=5:
                data.append(dat)
        if len(data)>0:
            data = pd.concat(data)
        else:
            data = pd.DataFrame()
        return data

    def to_meds_format(self, df, cohort_name):
        ids = list(df['person_id'].unique())
        grk = df.groupby('person_id')
        # time_index = grk['condition_start_date'].max().tolist()
        time_index = [grk.get_group(i)['condition_start_date'].max() for i in tqdm(ids)]
        df_meds = pd.DataFrame()
        df_meds['subject_id'] = ids
        df_meds['prediction_time'] = time_index
        if cohort_name =='case':
            df_meds['boolean_value']=1
        elif cohort_name =='ctrl':
            df_meds['boolean_value']=0
        return df_meds

In [ ]:
query = """
    SELECT c.concept_id, c.concept_name
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
"""
concept = pd.read_gbq(
    query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
concept_dict = dict(zip(concept['concept_id'], concept['concept_name']))

In [7]:
query = """
    SELECT co.person_id, co.condition_concept_id, co.condition_start_date
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` co
    WHERE co.condition_concept_id IN (
        SELECT ca.descendant_concept_id
        FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor` ca
        WHERE ca.ancestor_concept_id = 443392
        )
"""

cancer_diag = pd.read_gbq(
    query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

/tmp/ipykernel_245/1501490674.py:11: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  cancer_diag = pd.read_gbq(


Downloading:   0%|          |

In [11]:
query = """
SELECT DISTINCT co1.person_id
FROM `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` co1
WHERE NOT EXISTS (
    SELECT 1
    FROM `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` co2
    WHERE co2.person_id = co1.person_id
      AND co2.condition_concept_id IN (
          SELECT 443392
          UNION DISTINCT
          SELECT ca.descendant_concept_id
          FROM `""" + os.environ["WORKSPACE_CDR"] + """.concept_ancestor` ca
          WHERE ca.ancestor_concept_id = 443392
      )
)
"""
no_cancer = pd.read_gbq(
    query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

/tmp/ipykernel_245/1782241083.py:17: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  no_cancer = pd.read_gbq(


Downloading:   0%|          |

In [ ]:
cancer_diag = cancer_diag[~cancer_diag.duplicated()]
len(cancer_diag['person_id'].unique())

case = pd.DataFrame()
ctrl = pd.DataFrame()
cleaner = DataCleaner(case, ctrl, concept_dict = concept_dict, col_name = 'condition_concept_id')
cancer_cat = pd.read_csv('review_cancer_TL_cleaned.csv')
cancer_cat_aou = pd.read_csv('aou_cancer_categorization_TL.csv')
cancer_cat_aou = cancer_cat_aou.rename(columns = {'gpt category':'cancer_type', 'cancer name':'concept_name'})
cancer_cat = pd.concat([cancer_cat_aou[['concept_name','cancer_type']], cancer_cat[['concept_name','cancer_type']]])
cat_dict = dict(zip(cancer_cat['concept_name'], cancer_cat['cancer_type']))
cancer_diag['concept_name'] = cancer_diag['condition_concept_id'].map(concept_dict)
nan_lst, cancer_diag_cleaned = cleaner.cancer_categorization(cancer_diag, cancer_cat = cat_dict)
cancer_diag_cleaned = cleaner.exclude_dup_diag(cancer_diag_cleaned)
cleaner.first_cancer(cancer_diag_cleaned)
first_diag = cleaner.first_diag

In [ ]:
person_ids = first_diag['person_id'].tolist()
# person_ids = no_cancer['person_id'].tolist()

In [19]:
chunks = chunk_list(person_ids, 1000)

df_chunks = []
for i, chunk in tqdm(enumerate(chunks), total = len(chunks)):
    ids_str = ', '.join(str(int(pid)) for pid in chunk)  # ensures they're unquoted ints
    
    query = f"""
        SELECT c.person_id, c.condition_concept_id, c.condition_start_date
        FROM `{os.environ["WORKSPACE_CDR"]}.condition_occurrence` c
        WHERE c.person_id IN ({ids_str})
    """

    seq_data = pd.read_gbq(
        query,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ))
    
    df_chunks.append(seq_data)

  0%|          | 0/207 [00:00<?, ?it/s]/tmp/ipykernel_1961/2212169289.py:14: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  seq_data = pd.read_gbq(
100%|██████████| 207/207 [11:35<00:00,  3.36s/it]


In [20]:
len(df_chunks)

207

In [22]:
case_cleaned = []
for dat in tqdm(df_chunks):
    case = dat[dat['person_id'].isin(case_ids)]
    ctrl = []
    cleaner = DataCleaner(case, ctrl, 'condition_concept_id')
    
    case = cleaner.cancer_cohort(case_ids)
    case = cleaner.cancer_cohort_by_prediction_time(case, first_diag)
    
    case_cleaned.append(case)
case_cleaned_df = pd.concat(case_cleaned)
    
# ctrl_cleaned = []
# case = []
# for dat in tqdm(df_chunks):
#     ctrl = dat[dat['person_id'].isin(remaining_ctrl_ids)]
#     cleaner = DataCleaner(case, ctrl, 'condition_concept_id')
    
#     ctrl = cleaner.ctrl_cohort()
#     ctrl_cleaned.append(ctrl)
# ctrl_cleaned_df = pd.concat(ctrl_cleaned)

In [24]:
write_df_to_bucket(case_cleaned_df, 'case_cond')
# write_df_to_bucket(ctrl_cleaned_df, 'ctrl_cond')

# seq_data

In [26]:
def to_meds_format(df, cohort_name):
    ids = list(df['person_id'].unique())
    grk = df.groupby('person_id')
    # time_index = grk['condition_start_date'].max().tolist()
    time_index = [grk.get_group(i)['condition_start_date'].max() for i in tqdm(ids)]
    df_meds = pd.DataFrame()
    df_meds['subject_id'] = ids
    df_meds['prediction_time'] = time_index
    if cohort_name =='case':
        df_meds['boolean_value']=1
    elif cohort_name =='ctrl':
        df_meds['boolean_value']=0
    return df_meds
meds_case = to_meds_format(case_cleaned_df, 'case')
meds_ctrl = to_meds_format(ctrl_cleaned_df, 'ctrl')

100%|██████████| 37893/37893 [00:18<00:00, 2021.70it/s]


In [46]:
meds_all_data = pd.concat([meds_case, meds_ctrl])
write_df_to_bucket(meds_all_data, 'meds_all_data')

In [ ]:
cancer_type = 'pancreas'
case_ids = first_diag[first_diag['cancer_type']==cancer_type]['person_id'].tolist()
case_meds == meds_data[meds_data['boolean_value']==1]
specific_case_meds = case_meds[case_meds['subject_id'].isin(case_ids)]
ctrl_meds = meds_data[meds_data['boolean_value']==0]
meds_data = pd.concat([specific_case_meds, ctrl_meds])

In [28]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Generate splits
data_fold= {}
for fold, (train_idx, test_idx) in enumerate(skf.split(meds_data['subject_id'], meds_data['boolean_value'])):
    train_df = meds_ctrl.iloc[train_idx]
    test_df = meds_ctrl.iloc[test_idx]
    
    data_fold[fold] = {}
    data_fold[fold]['train'] = train_df['subject_id'].tolist()
    data_fold[fold]['test'] = test_df['subject_id'].tolist()

In [30]:
def seq_data_with_value(df, meds_data):
    cols = list(df.columns)
    col_date = [i for i in cols if 'date' in i][0]
    col_concept_id = [i for i in cols if 'concept' in i][0]

    df = df[['person_id', col_date, col_concept_id]]
    df = df[~df.duplicated()]
    df = df[df[col_concept_id]!=0]

    df[col_date] = pd.to_datetime(df[col_date])
    df = df.sort_values(by = col_date)
    grk = df.groupby('person_id')
#     ids = meds_data['subject_id'].tolist()
    ids = list(df['person_id'].unique())


    prediction_time = dict(zip(meds_data['subject_id'], meds_data['prediction_time']))

    label_dict = dict(zip(meds_data['subject_id'], meds_data['boolean_value']))
    split_dict = dict(zip(meds_data['subject_id'], meds_data['split']))
    data_dict = {}
    for i in tqdm(ids):
        data_dict[i] = {}
        pat_dat = grk.get_group(i)
        pat_dat = pat_dat[pat_dat[col_date]<=prediction_time[i]]
        data_dict[i]['seq'] = pat_dat[col_concept_id].tolist()
        data_dict[i]['time'] = [(prediction_time[i]-i_time).days/365.25 for i_time in pat_dat[col_date].tolist()]
        data_dict[i]['label'] = int(label_dict[i])
        data_dict[i]['split'] = split_dict[i]
    return data_dict

In [33]:
for fold, ids in data_fold.items():
    train_ids = set(ids['train'])  # make lookup fast
    test_ids = set(ids['test'])
    all_ids = train_ids.union(test_ids)

    # Filter meds_data by subject_id only
    sub_meds = meds_ctrl[meds_ctrl['subject_id'].isin(all_ids)].copy()

    # Assign split using vectorized Pandas logic
    sub_meds['split'] = sub_meds['subject_id'].apply(lambda x: 'train' if x in train_ids else 'test')

    # Filter df similarly
    sub_df = ctrl_cleaned_df[ctrl_cleaned_df['person_id'].isin(all_ids)]

    data_dict = seq_data_with_value(sub_df, sub_meds)
    save_dict(data_dict, f'seq_data_{cancer_type}_condition'+str(fold))

100%|██████████| 37893/37893 [01:01<00:00, 620.64it/s]
